# Reentrenamiento con el dataset v2 de Roboflow (best.pt v2)

Dataset: `Deteccion de Sillar v2` (Instance Segmentation, 2030 imágenes, split 70/20/10,
*Fit within 640*), ya convertido a las 3 clases de la app con `preparar_dataset_v2.py`:

| Roboflow | App |
|---|---|
| `grieta`, `fisura` | `crack` |
| `humedad`, `humeldad` | `humidity` |
| `desprendimiento` | `spalling` |
| `Patologias`, `3` | imagen excluida (29 imágenes) |

En train se duplican ×3 las imágenes con grietas (desbalance humidity/crack de 8.4:1 → 3.6:1).
Valid y test no se tocan, así que las métricas no están infladas.

## Pasos
1. **Entorno de ejecución → Cambiar tipo de entorno → GPU T4.**
2. Sube `dataset_3clases.zip` a la raíz de tu Google Drive (*Mi unidad*).
3. Ejecuta las celdas en orden. El entrenamiento se guarda **en Drive**: si Colab se desconecta,
   ejecuta las celdas 1, 2 y la celda **5 (reanudar)**; no se pierde lo avanzado.

Tiempo estimado en T4: ~1.5–3 h (100 épocas, se detiene antes si no mejora en 20).

In [ ]:
# 1) Montar Google Drive y verificar GPU
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2) Instalar la misma version de ultralytics usada para best.pt v1
!pip install -q ultralytics==8.4.56

In [ ]:
# 3) Descomprimir el dataset y comprobarlo
import os, zipfile, glob
import pandas as pd

ZIP = "/content/drive/MyDrive/dataset_3clases.zip"
DESTINO = "/content/datasets/sillar_3clases"
assert os.path.exists(ZIP), f"No encuentro {ZIP}: sube dataset_3clases.zip a la raiz de Mi unidad"
if not os.path.exists(f"{DESTINO}/data.yaml"):
    zipfile.ZipFile(ZIP).extractall(DESTINO)
print(open(f"{DESTINO}/data.yaml").read())

for split in ("train", "valid", "test"):
    print(f"{split:6s} {len(os.listdir(f'{DESTINO}/{split}/images')):5d} imagenes")
lbl = next(p for p in glob.glob(f"{DESTINO}/train/labels/*.txt") if os.path.getsize(p))
es_segmentacion = len(open(lbl).readline().split()) > 5
print("Etiquetas:", "SEGMENTACION (poligonos)" if es_segmentacion else "CAJAS")
print(open(f"{DESTINO}/distribucion_dataset_v2.csv").read().split("\n\n")[0])

In [ ]:
# 4) ENTRENAMIENTO (mismos hiperparametros que best.pt v1 + augmentations del problema real)
from ultralytics import YOLO

TAREA = "segment" if es_segmentacion else "detect"
PESOS = "yolov8s-seg.pt" if TAREA == "segment" else "yolov8s.pt"
PROYECTO = "/content/drive/MyDrive/heritage_runs"   # en Drive: sobrevive a desconexiones
NOMBRE = "heritage_damage_v2"

model = YOLO(PESOS)
model.train(
    data=f"{DESTINO}/data.yaml",
    epochs=100,
    patience=20,
    batch=16,
    imgsz=640,
    device=0,
    workers=8,
    project=PROYECTO,
    name=NOMBRE,
    exist_ok=True,
    optimizer="AdamW",
    seed=42,
    deterministic=True,
    save_period=10,
    hsv_v=0.2,      # brillo +-20%: una sombra no es una grieta
    degrees=10.0,   # rotacion +-10 grados
    fliplr=0.5,     # flip horizontal
    flipud=0.0,     # sin flip vertical: la gravedad importa (humedad, desprendimiento)
)
BEST = str(model.trainer.best)
print("Mejor modelo:", BEST)

In [ ]:
# 5) SOLO SI COLAB SE DESCONECTO: reanudar desde el ultimo checkpoint en Drive
#    (ejecuta antes las celdas 1, 2 y 3)
import glob
from ultralytics import YOLO
last = sorted(glob.glob("/content/drive/MyDrive/heritage_runs/**/weights/last.pt", recursive=True))[-1]
print("Reanudando:", last)
model = YOLO(last)
model.train(resume=True)
BEST = str(model.trainer.best)
TAREA = model.task
print("Mejor modelo:", BEST)

In [ ]:
# 6) Metricas en TEST (tabla para la tesis) y verificacion de clases
import pandas as pd
from ultralytics import YOLO

tuned = YOLO(BEST)
assert tuned.names == {0: "crack", 1: "humidity", 2: "spalling"}, tuned.names
print("Clases OK:", tuned.names, "| tarea:", tuned.task)

m = tuned.val(data=f"{DESTINO}/data.yaml", split="test", conf=0.001, iou=0.45,
              project="/content/drive/MyDrive/heritage_runs", name="test_v2", exist_ok=True)
filas = []
for k, i in enumerate(m.box.ap_class_index):   # solo clases presentes en test
    p, r, ap50, ap = m.box.class_result(k)
    fila = {"clase": tuned.names[int(i)], "precision": p, "recall": r, "mAP50": ap50, "mAP50-95": ap}
    if tuned.task == "segment":
        fila["mask_mAP50"] = m.seg.class_result(k)[2]
        fila["mask_mAP50-95"] = m.seg.class_result(k)[3]
    filas.append(fila)
metricas = pd.DataFrame(filas)
print(metricas.round(3).to_string(index=False))
print(m.results_dict)
metricas.to_csv("/content/drive/MyDrive/metricas_test_v2.csv", index=False)
print("Graficas (matriz de confusion, curvas PR):", m.save_dir)

In [ ]:
# 7) Copiar best.pt al Drive y descargarlo
#    Luego: renombra el best.pt viejo (best_v1.pt) y copia este a Heritage_Detector\best.pt
import shutil
shutil.copy(BEST, "/content/drive/MyDrive/best_v2.pt")
from google.colab import files
files.download(BEST)